<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/lotus_online_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# Install necessary packages for Selenium in Colab
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium bs4 pandas
!pip install playwright beautifulsoup4 polars
!playwright install chromium
!playwright install-deps

# 1. Clear out the broken Ubuntu packages
!apt-get remove -y chromium-browser chromium-chromedriver

# 2. Download and install the official Google Chrome stable version
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

# 3. Install Python dependencies including the auto-installer
!pip install selenium bs4 pandas chromedriver-autoinstaller
!pip install polars xlsxwriter fastexcel
!pip install playwright beautifulsoup4 polars
!playwright install
!pip install itables
!playwright install-deps
!pip install curl_cffi

%%capture
# 1. Install all dependencies including pandas
!pip install xlsxwriter scrapling patchright msgspec browserforge nest_asyncio polars -q
!patchright install chromium
!patchright install-deps

### Import Lib

In [ ]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

In [ ]:
# @title udf scrape lotus
###
# --- This code work
###

async def scrape_lotuss_scroller(shop_url: str):
    extracted_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        print(f"Opening shop... Please wait.")
        await page.goto(shop_url, wait_until="domcontentloaded")

        try:
            # Wait for the first batch of products to load
            await page.wait_for_selector(".mui-style-17swnep", timeout=15000)
        except Exception as e:
            print("Failed to load the initial products.")
            await browser.close()
            return pl.DataFrame()

        # --- THE INFINITE SCROLLER ---
        previous_item_count = 0
        scroll_attempts = 0
        max_scrolls = 30 # Safety net so it doesn't scroll literally forever

        while scroll_attempts < max_scrolls:
            # Tell the browser to scroll to the absolute bottom of the page
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")

            # Wait 4 seconds for Lotus's servers to fetch and draw the next batch of products
            await asyncio.sleep(4)

            # Count how many products are currently physically on the screen
            current_items = await page.query_selector_all(".MuiCard-root")
            current_count = len(current_items)

            print(f"Scroll {scroll_attempts + 1}: Found {current_count} products on screen...")

            # If we scrolled down but the number of products didn't increase, we hit the bottom!
            if current_count == previous_item_count:
                print("Hit the bottom of the catalog! Scraping all loaded data...")
                break

            previous_item_count = current_count
            scroll_attempts += 1

        # Now that the browser has fully expanded all products, take a snapshot of the HTML
        html_content = await page.content()
        soup = BeautifulSoup(html_content, "html.parser")
        products = soup.find_all("div", class_="MuiCard-root")

        for prod in products:
            # 1. Extract Name
            name_elem = prod.find(class_="mui-style-17swnep")
            name = name_elem.text.strip() if name_elem else None

            if not name:
                continue

            # 2. Extract Promotion Price
            promo_elem = prod.find("div", class_="mui-style-18s6ztp")
            if promo_elem:
                raw_promo = promo_elem.text.replace(',', '').replace('฿', '').strip()
                promotion_price = raw_promo if raw_promo else None
            else:
                promotion_price = None

            # 3. Extract Original Price
            orig_elem = prod.find("div", class_="mui-style-f59hd5")
            if orig_elem:
                raw_orig = orig_elem.text.replace(',', '').replace('฿', '').strip()
                original_price = raw_orig if raw_orig else promotion_price
            else:
                original_price = promotion_price

            extracted_data.append({
                "name": name,
                "promotion_price": promotion_price,
                "original_price": original_price
            })

        await browser.close()

    df = pl.DataFrame(extracted_data)

    # Final cleanup of any accidental duplicates
    if not df.is_empty():
        df = df.unique(subset=["name"], maintain_order=True)

    return df

In [ ]:
# @title RUN create dataframe
# --- RUN IT ---
# Notice we are just passing the raw URL without trying to add "&page="
shop_laundry_supplies_url = "https://www.lotuss.com/en/category/household-and-merits/86590-laundry-supplies?sort=relevance:DESC&filter.brandId=21490,21430,21829,22054,23485"
df_laundry_supplies = await scrape_lotuss_scroller(shop_url=shop_laundry_supplies_url)

print("\n--- Scraping Complete ---")
print(df_laundry_supplies)


shop_dish_washing_liquid_url = "https://www.lotuss.com/en/category/household-and-merits/86590-cleaning-chemical/86671-dish-washing-liquid?sort=relevance:DESC&filter.brandId=22327"
df_lotuss_dish_washing_liquid = await scrape_lotuss_scroller(shop_url=shop_dish_washing_liquid_url)

print("\n--- Scraping Complete ---")
print(df_lotuss_dish_washing_liquid)

In [ ]:
df_lotuss = pl.concat([df_lotuss_dish_washing_liquid, df_laundry_supplies])

def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# How to use it:
df_prep_lotuss = re_evaluate_price(df_lotuss)

In [ ]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(date.today()).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int64, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [ ]:
# 2. Transform Lotus's (from yesterday's script)
df_trans_lotuss = parse_product_names(df_prep_lotuss, shop_name = "Lotus's")

In [ ]:
# @title write Excel
df_trans_lotuss.write_excel(f"lotus_{today_date}.xlsx")